[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/30.%20The%20Yard%20Pre-Marshalling%20for%20Exports%20Problem/P30-Tier-1-SOLUTION_executed.ipynb)

---

# 30. The Yard Pre-Marshalling for Exports Problem
## Tier 1: Position-Network Representation and Exact Enumeration

### Goal
Find a final assignment of containers to yard positions that satisfies the priority-ordering rule while minimizing the number of containers assigned to a different position. The notebook also constructs a directed position-transition network to illustrate possible relocations and compares the exact assignment search with a simple greedy assignment.

### Inputs and outputs
The input is a set of containers, each with an identifier, a priority, and an initial stack and tier, together with the number of stacks and tiers. Lower numeric priority values mean higher retrieval priority. The exact search returns a feasible final position for every container and the minimum relocation count under the static assignment model. The notebook reports the initial and final priority violations and displays the configurations.

### Model assumptions and scope
- Each container occupies one distinct position, and each position holds at most one container.
- Tiers are indexed from the bottom upward. Within a stack, the priorities of its occupied positions must be nondecreasing as tier increases; empty positions are allowed in this abstract assignment model.
- Every container assigned to a position other than its initial position contributes one unit to the objective. This is a relocation-count proxy, not a count of executable crane lifts or a complete move sequence.
- The model does not represent container accessibility, temporary storage, gravity or compact stacking, blocking, crane travel, or operational timing. The position-transition network is illustrative; the exact solution is obtained by enumerating assignments, not by solving a minimum-cost-flow model.

### Mathematical model and exact search
Let $C$ be the set of containers and $P$ the set of all stack-tier positions. For container $c \in C$, let $p_c^0 \in P$ denote its initial position, and let $\pi_c$ be its priority. An assignment $a_c \in P$ must give distinct positions to distinct containers. In each stack, when the occupied positions are read from the bottom upward, their priorities must be nondecreasing.

The objective is

$$
\min_{(a_c)_{c \in C}} \quad \sum_{c \in C} \mathbf{1}[a_c \ne p_c^0],
$$

subject to distinct assigned positions and the stack-ordering rule. Here, $\mathbf{1}[a_c \ne p_c^0]$ equals 1 if container $c$ changes position and 0 otherwise. The function `calculate_move_cost` evaluates this objective, and `is_valid_assignment` checks the feasibility conditions.

The exact procedure enumerates ordered selections of $|C|$ distinct positions from $|P|$ positions. It checks every candidate assignment, retains a feasible assignment with the smallest objective value, and therefore finds a global optimum for this finite model. The number of candidates is $|P|!/(|P|-|C|)!$; this grows rapidly, so the method is intended for small instances. If multiple assignments share the minimum cost, the enumeration returns the first one encountered.

### Position-transition network
The directed graph has a source, one node for each yard position, and a sink. Source-to-position arcs indicate initial positions, arcs between distinct position nodes represent possible relocations, and position-to-sink arcs complete the illustrative graph. The edge costs are descriptive: the graph does not encode container-specific assignments or the stack-ordering constraints, and it is not the optimization engine used in this notebook. The exact search above is the method that determines the reported optimum.

### Concrete example
The example has two stacks, three tiers per stack, and four containers:
- Stack 0: container A (priority 3) at tier 0 and container B (priority 1) at tier 1.
- Stack 1: container C (priority 4) at tier 0 and container D (priority 2) at tier 1.

Because lower numbers indicate higher priority, each initially occupied stack has a priority violation. The code evaluates feasible final assignments rather than assuming a particular relocation sequence.

In [ ]:
import itertools
from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("default")

In [ ]:
@dataclass
class Container:
    """A container with a priority and an initial bay position."""

    id: str
    priority: int  # Lower values indicate higher retrieval priority.
    current_stack: int
    current_tier: int


@dataclass
class Position:
    """A stack-tier position in the container bay."""

    stack: int
    tier: int


@dataclass
class NetworkFlowModel:
    """Illustrative position-transition network for the assignment example."""

    containers: List[Container]
    stacks: int
    tiers: int

    def __post_init__(self):
        self.positions = [
            (stack, tier)
            for stack in range(self.stacks)
            for tier in range(self.tiers)
        ]
        self.pos_to_idx = {
            position: index for index, position in enumerate(self.positions)
        }
        self.n_positions = len(self.positions)

        self.source = "source"
        self.sink = "sink"
        self.nodes = [self.source] + [
            f"pos_{stack}_{tier}" for stack, tier in self.positions
        ] + [self.sink]

        self.adjacency = defaultdict(list)
        self.costs = {}
        self._build_network()

    def _build_network(self):
        """Build arcs that illustrate initial positions and possible relocations."""
        initial_positions = {
            (container.current_stack, container.current_tier)
            for container in self.containers
        }
        for stack, tier in sorted(initial_positions):
            from_node = self.source
            to_node = f"pos_{stack}_{tier}"
            self.adjacency[from_node].append(to_node)
            self.costs[(from_node, to_node)] = 0

        for from_stack, from_tier in self.positions:
            from_node = f"pos_{from_stack}_{from_tier}"
            for to_stack, to_tier in self.positions:
                if (from_stack, from_tier) != (to_stack, to_tier):
                    to_node = f"pos_{to_stack}_{to_tier}"
                    self.adjacency[from_node].append(to_node)
                    self.costs[(from_node, to_node)] = 1

        for stack, tier in self.positions:
            from_node = f"pos_{stack}_{tier}"
            self.adjacency[from_node].append(self.sink)
            self.costs[(from_node, self.sink)] = 0

In [ ]:
def check_priority_violation(
    containers: List[Container], stacks: int, tiers: int
) -> int:
    """Count adjacent priority inversions in the occupied order of each stack."""
    priorities_by_stack = [[] for _ in range(stacks)]
    for container in containers:
        priorities_by_stack[container.current_stack].append(
            (container.current_tier, container.priority)
        )

    violations = 0
    for stack_priorities in priorities_by_stack:
        ordered_priorities = [
            priority
            for _, priority in sorted(stack_priorities, key=lambda item: item[0])
        ]
        violations += sum(
            lower_priority > upper_priority
            for lower_priority, upper_priority in zip(
                ordered_priorities, ordered_priorities[1:]
            )
        )

    return violations


def is_valid_assignment(
    containers: List[Container],
    assignments: Dict[str, Tuple[int, int]],
    stacks: int,
    tiers: int,
) -> bool:
    """Check container coverage, position capacity, and stack priority order."""
    container_ids = {container.id for container in containers}
    if set(assignments) != container_ids:
        return False

    occupied_positions = list(assignments.values())
    if len(set(occupied_positions)) != len(occupied_positions):
        return False

    for stack, tier in occupied_positions:
        if not (0 <= stack < stacks and 0 <= tier < tiers):
            return False

    priority_by_id = {container.id: container.priority for container in containers}
    priorities_by_stack = [[] for _ in range(stacks)]
    for container_id, (stack, tier) in assignments.items():
        priorities_by_stack[stack].append((tier, priority_by_id[container_id]))

    for stack_priorities in priorities_by_stack:
        ordered_priorities = [
            priority
            for _, priority in sorted(stack_priorities, key=lambda item: item[0])
        ]
        if any(
            lower_priority > upper_priority
            for lower_priority, upper_priority in zip(
                ordered_priorities, ordered_priorities[1:]
            )
        ):
            return False

    return True


def calculate_move_cost(
    initial_assignments: Dict[str, Tuple[int, int]],
    final_assignments: Dict[str, Tuple[int, int]],
) -> int:
    """Count containers whose final position differs from their initial one."""
    return sum(
        initial_assignments[container_id] != final_assignments[container_id]
        for container_id in initial_assignments
    )

In [ ]:
def solve_network_flow_exact(
    model: NetworkFlowModel,
) -> Tuple[Dict[str, Tuple[int, int]], int]:
    """Enumerate assignments and return a minimum-cost feasible assignment."""
    containers = model.containers
    all_positions = model.positions
    initial_assignment = {
        container.id: (container.current_stack, container.current_tier)
        for container in containers
    }

    best_assignment = None
    min_cost = float("inf")

    for assignment_positions in itertools.permutations(
        all_positions, len(containers)
    ):
        assignment = {
            containers[index].id: assignment_positions[index]
            for index in range(len(containers))
        }

        if is_valid_assignment(
            containers, assignment, model.stacks, model.tiers
        ):
            cost = calculate_move_cost(initial_assignment, assignment)
            if cost < min_cost:
                min_cost = cost
                best_assignment = assignment

    if best_assignment is None:
        raise ValueError("No feasible assignment exists for this instance.")

    return best_assignment, min_cost


def solve_network_flow_greedy(
    model: NetworkFlowModel,
) -> Dict[str, Tuple[int, int]]:
    """Assign containers by priority to the lowest available tiers."""
    sorted_containers = sorted(model.containers, key=lambda container: container.priority)
    occupied = set()
    assignment = {}

    for container in sorted_containers:
        best_position = None
        min_tier = float("inf")

        for stack in range(model.stacks):
            for tier in range(model.tiers):
                if (stack, tier) not in occupied and tier < min_tier:
                    min_tier = tier
                    best_position = (stack, tier)

        if best_position is not None:
            assignment[container.id] = best_position
            occupied.add(best_position)

    return assignment

In [ ]:
def visualize_bay_configuration(
    containers: List[Container], stacks: int, tiers: int, title: str
):
    """Plot container positions and show the priority-to-color mapping."""
    fig, ax = plt.subplots(figsize=(10, 6))
    priority_values = [container.priority for container in containers]
    minimum_priority = min(priority_values)
    maximum_priority = max(priority_values)
    if minimum_priority == maximum_priority:
        minimum_priority -= 0.5
        maximum_priority += 0.5
    norm = plt.Normalize(minimum_priority, maximum_priority)
    cmap = plt.cm.RdYlBu_r

    grid = [[None for _ in range(tiers)] for _ in range(stacks)]
    for container in containers:
        grid[container.current_stack][container.current_tier] = container

    for stack in range(stacks):
        for tier in range(tiers):
            container = grid[stack][tier]
            facecolor = (
                cmap(norm(container.priority)) if container is not None else "lightgray"
            )
            edgecolor = "black" if container is not None else "gray"
            rectangle = plt.Rectangle(
                (stack * 1.2, tier * 0.8),
                1.0,
                0.6,
                facecolor=facecolor,
                edgecolor=edgecolor,
                linewidth=2 if container is not None else 1,
                alpha=1.0 if container is not None else 0.3,
            )
            ax.add_patch(rectangle)

            if container is not None:
                ax.text(
                    stack * 1.2 + 0.5,
                    tier * 0.8 + 0.3,
                    f"{container.id}\n(Priority {container.priority})",
                    ha="center",
                    va="center",
                    fontweight="bold",
                    fontsize=9,
                )

    ax.set_xlim(-0.5, stacks * 1.2)
    ax.set_ylim(-0.5, tiers * 0.8)
    ax.set_xlabel("Stack index")
    ax.set_ylabel("Tier index (bottom to top)")
    ax.set_title(title, fontweight="bold")
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)

    for stack in range(stacks):
        ax.text(stack * 1.2 + 0.5, -0.3, f"S{stack}", ha="center", fontweight="bold")
    for tier in range(tiers):
        ax.text(-0.3, tier * 0.8 + 0.3, f"T{tier}", ha="center", fontweight="bold")

    colorbar = fig.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, fraction=0.046, pad=0.04
    )
    colorbar.set_label("Priority value (lower means higher priority)")
    plt.tight_layout()
    plt.show()


def visualize_solution_comparison(
    initial_containers: List[Container],
    final_assignment: Dict[str, Tuple[int, int]],
    stacks: int,
    tiers: int,
):
    """Plot initial and assigned configurations with a shared priority scale."""
    final_containers = [
        Container(
            container.id,
            container.priority,
            final_assignment[container.id][0],
            final_assignment[container.id][1],
        )
        for container in initial_containers
    ]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    priority_values = [container.priority for container in initial_containers]
    minimum_priority = min(priority_values)
    maximum_priority = max(priority_values)
    if minimum_priority == maximum_priority:
        minimum_priority -= 0.5
        maximum_priority += 0.5
    norm = plt.Normalize(minimum_priority, maximum_priority)
    cmap = plt.cm.RdYlBu_r

    def draw_bay(ax, bay_containers, title):
        grid = [[None for _ in range(tiers)] for _ in range(stacks)]
        for container in bay_containers:
            grid[container.current_stack][container.current_tier] = container

        for stack in range(stacks):
            for tier in range(tiers):
                container = grid[stack][tier]
                facecolor = (
                    cmap(norm(container.priority))
                    if container is not None
                    else "lightgray"
                )
                edgecolor = "black" if container is not None else "gray"
                rectangle = plt.Rectangle(
                    (stack * 1.2, tier * 0.8),
                    1.0,
                    0.6,
                    facecolor=facecolor,
                    edgecolor=edgecolor,
                    linewidth=2 if container is not None else 1,
                    alpha=1.0 if container is not None else 0.3,
                )
                ax.add_patch(rectangle)

                if container is not None:
                    ax.text(
                        stack * 1.2 + 0.5,
                        tier * 0.8 + 0.3,
                        f"{container.id}\n(Priority {container.priority})",
                        ha="center",
                        va="center",
                        fontweight="bold",
                        fontsize=9,
                    )

        ax.set_xlim(-0.5, stacks * 1.2)
        ax.set_ylim(-0.5, tiers * 0.8)
        ax.set_xlabel("Stack index")
        ax.set_ylabel("Tier index (bottom to top)")
        ax.set_title(title, fontweight="bold")
        ax.set_aspect("equal")
        ax.grid(True, alpha=0.3)

        for stack in range(stacks):
            ax.text(
                stack * 1.2 + 0.5, -0.3, f"S{stack}", ha="center", fontweight="bold"
            )
        for tier in range(tiers):
            ax.text(-0.3, tier * 0.8 + 0.3, f"T{tier}", ha="center", fontweight="bold")

    draw_bay(ax1, initial_containers, "Initial Configuration")
    draw_bay(ax2, final_containers, "Minimum-Relocation Assignment")
    colorbar = fig.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap=cmap),
        ax=[ax1, ax2],
        fraction=0.025,
        pad=0.04,
    )
    colorbar.set_label("Priority value (lower means higher priority)")
    plt.tight_layout()
    plt.show()

In [ ]:
print("=" * 60)
print("YARD PRE-MARSHALLING: POSITION ASSIGNMENT EXAMPLE")
print("=" * 60)

containers = [
    Container("A", 3, 0, 0),
    Container("B", 1, 0, 1),
    Container("C", 4, 1, 0),
    Container("D", 2, 1, 1),
]
stacks, tiers = 2, 3

assert len({container.id for container in containers}) == len(containers)
assert len(containers) <= stacks * tiers
assert all(
    0 <= container.current_stack < stacks
    and 0 <= container.current_tier < tiers
    for container in containers
)
assert len(
    {(container.current_stack, container.current_tier) for container in containers}
) == len(containers)

print("\nProblem configuration:")
print(f"- Stacks: {stacks}")
print(f"- Tiers per stack: {tiers}")
print(f"- Containers: {len(containers)}")

print("\nInitial container positions:")
for container in containers:
    print(
        f"  Container {container.id}: priority {container.priority}, "
        f"stack {container.current_stack}, tier {container.current_tier}"
    )

initial_violations = check_priority_violation(containers, stacks, tiers)
print(f"\nInitial adjacent priority violations: {initial_violations}")

visualize_bay_configuration(
    containers, stacks, tiers, "Initial Bay Configuration"
)

In [ ]:
model = NetworkFlowModel(containers, stacks, tiers)

print("\n" + "=" * 60)
print("ILLUSTRATIVE POSITION-TRANSITION NETWORK")
print("=" * 60)
print(f"\nNetwork nodes: {len(model.nodes)}")
print(f"- Source node: {model.source}")
print(f"- Position nodes: {model.n_positions}")
print(f"- Sink node: {model.sink}")
print("\nListed arcs and descriptive costs:")

edge_count = 0
for from_node, to_nodes in model.adjacency.items():
    for to_node in to_nodes:
        cost = model.costs[(from_node, to_node)]
        if from_node != model.source or len(to_nodes) <= 5:
            print(f"  {from_node} -> {to_node}: cost = {cost}")
        edge_count += 1

print(f"\nTotal arcs in the illustrative network: {edge_count}")
print("The exact assignment search, not this graph, determines the optimum.")

In [ ]:
print("\n" + "=" * 60)
print("EXACT ASSIGNMENT ENUMERATION")
print("=" * 60)

optimal_assignment, min_cost = solve_network_flow_exact(model)
assert is_valid_assignment(containers, optimal_assignment, stacks, tiers)

initial_assignment = {
    container.id: (container.current_stack, container.current_tier)
    for container in containers
}
assert calculate_move_cost(initial_assignment, optimal_assignment) == min_cost

print(f"\nMinimum number of relocated containers: {min_cost}")
print("\nMinimum-cost final assignment:")
for container_id, (stack, tier) in optimal_assignment.items():
    print(f"  Container {container_id}: stack {stack}, tier {tier}")

final_containers = [
    Container(
        container.id,
        container.priority,
        optimal_assignment[container.id][0],
        optimal_assignment[container.id][1],
    )
    for container in containers
]
final_violations = check_priority_violation(final_containers, stacks, tiers)
print(f"\nFinal adjacent priority violations: {final_violations}")
assert final_violations == 0
print("The assigned configuration satisfies the priority-ordering rule.")

visualize_solution_comparison(containers, optimal_assignment, stacks, tiers)

In [ ]:
print("\n" + "=" * 60)
print("COMPARISON WITH A GREEDY ASSIGNMENT")
print("=" * 60)

greedy_assignment = solve_network_flow_greedy(model)
assert is_valid_assignment(containers, greedy_assignment, stacks, tiers)
greedy_cost = calculate_move_cost(initial_assignment, greedy_assignment)

print(f"\nGreedy relocation count: {greedy_cost}")
print("\nGreedy assignment:")
for container_id, (stack, tier) in greedy_assignment.items():
    print(f"  Container {container_id}: stack {stack}, tier {tier}")

greedy_containers = [
    Container(
        container.id,
        container.priority,
        greedy_assignment[container.id][0],
        greedy_assignment[container.id][1],
    )
    for container in containers
]
greedy_violations = check_priority_violation(greedy_containers, stacks, tiers)

print(f"\nGreedy adjacent priority violations: {greedy_violations}")
print("\nAssignment comparison:")
print(f"{'Method':<30} {'Relocations':<15} {'Violations':<12}")
print("-" * 60)
print(f"{'Exact enumeration':<30} {min_cost:<15} {final_violations:<12}")
print(f"{'Greedy assignment':<30} {greedy_cost:<15} {greedy_violations:<12}")

print(
    "\nThe exact enumeration establishes the minimum relocation count "
    "for the stated finite assignment model. The greedy result is a "
    "feasible comparison, not an optimality certificate."
)

In [ ]:
print("\n" + "=" * 60)
print("ASSIGNMENT AND STACK ANALYSIS")
print("=" * 60)

print("\nRelocation pattern:")
moved_containers = []
for container in containers:
    initial_position = (container.current_stack, container.current_tier)
    final_position = optimal_assignment[container.id]
    if initial_position != final_position:
        moved_containers.append((container.id, initial_position, final_position))
        print(f"  Container {container.id}: {initial_position} -> {final_position}")

print(f"\nContainers assigned to a different position: {len(moved_containers)}")
print(
    "Relocation fraction: "
    f"{len(moved_containers)}/{len(containers)} = "
    f"{len(moved_containers) / len(containers):.2%}"
)

print("\nFinal stack utilization and order:")
for stack in range(stacks):
    stack_containers = sorted(
        [
            container
            for container in final_containers
            if container.current_stack == stack
        ],
        key=lambda container: container.current_tier,
    )
    priorities = [container.priority for container in stack_containers]
    is_ordered = all(
        lower <= upper for lower, upper in zip(priorities, priorities[1:])
    )
    status = "Ordered" if is_ordered else "Not ordered"
    print(
        f"  Stack {stack}: {len(stack_containers)} containers, "
        f"bottom-to-top priorities {priorities}; {status}"
    )

all_priorities = [container.priority for container in final_containers]
print("\nPriority summary:")
print(f"  Priority range: {min(all_priorities)} to {max(all_priorities)}")
print(f"  Mean priority: {np.mean(all_priorities):.2f}")
print(f"  Priority variance: {np.var(all_priorities):.2f}")

## Interpretation, strengths, and limitations

### What the exact result means
The exact enumeration examines every distinct-position assignment in the stated model and retains a feasible assignment with the lowest relocation-count objective. Its optimality guarantee applies to this finite static assignment model: it does not establish that the assignment can be reached through a legal sequence of container lifts. The greedy assignment provides a feasible comparison, while its result alone does not prove optimality.

### Strengths
- The objective and priority-ordering rule are explicit and can be checked directly for every returned assignment.
- Exhaustive enumeration provides a global optimum for instances small enough to search completely.
- The assignment representation makes it straightforward to inspect which containers change positions and how each stack is ordered.

### Limitations
- The number of candidate assignments grows as $|P|!/(|P|-|C|)!$, so exhaustive enumeration becomes impractical as the number of containers and positions increases.
- The relocation-count objective is not an operational lift count or a complete handling-cost model.
- The static constraints omit access, blocking, temporary moves, compact stack occupancy, crane operations, and timing.
- The displayed position-transition graph is a structural illustration. It does not by itself encode the container-specific assignment decisions, priority-ordering constraints, or exact-search procedure.

### Scope of extensions
A broader operational model could specify additional physical constraints, handling costs, or move-sequence requirements. Those features are outside the static assignment model implemented here.